# MIA Summary Explorer
Interactively group and compare rows in results/summary_by_group.csv.

- Group on the same (dataset, sparsity, attack, metric)
- Compare across (method, mode) with stats: mean, std, CI, n (unique victims)


In [3]:
import pandas as pd
from pathlib import Path
from IPython.display import display

CSV_PATH = Path('../results/summary_by_group.csv')
assert CSV_PATH.exists(), f'CSV not found: {CSV_PATH}'
df = pd.read_csv(CSV_PATH)
if 'sparsity' in df.columns:
    df['sparsity'] = pd.to_numeric(df['sparsity'], errors='coerce')
if 'mode' not in df.columns:
    df['mode'] = ''
display(df.head())


,dataset,method,mode,sparsity,attack,metric,use_temperature,mean,std,ci95_lo,ci95_hi,n,median,iqr
0,cifar10,dense,dense,0.0,confidence,advantage,NaN,0.109818,0.010524,0.105097,0.114743,17,0.108800,0.012356
1,cifar10,dpf,dpf:freeze180,0.5,confidence,advantage,NaN,0.098663,0.014028,0.092470,0.105403,17,0.094978,0.015022
2,cifar10,dpf,dpf:nofreeze,0.5,confidence,advantage,NaN,0.093265,0.009608,0.088740,0.097724,17,0.092422,0.009667
3,cifar10,dwa,kill_active_plain_dead,0.5,confidence,advantage,NaN,0.088941,0.014706,0.082261,0.095647,17,0.084622,0.017889
4,cifar10,dwa,kill_and_reactivate,0.5,confidence,advantage,NaN,0.091328,0.015715,0.084098,0.098562,17,0.089800,0.022378


In [ ]:
import ipywidgets as W
from IPython.display import display, HTML

def pivot_block(dataset, sparsity, attack, metric, sort_by='mean', ascending=False):
    sub = df.copy()
    if dataset is not None:
        sub = sub[sub['dataset'] == dataset]
    sub = sub[(sub['sparsity'] == sparsity) & (sub['attack'] == attack) & (sub['metric'] == metric)]
    if sub.empty:
        return pd.DataFrame(), pd.DataFrame()
    cols = ['mean','std','ci95_lo','ci95_hi','n','median','iqr']
    show = ['dataset','sparsity','attack','metric','method','mode'] + cols
    sub = sub[show].copy()
    pvt = (sub.set_index(['method','mode'])[cols]
              .sort_values(by=[sort_by], ascending=ascending))
    return pvt, sub.sort_values(by=[sort_by], ascending=ascending)

datasets   = sorted(df['dataset'].dropna().unique().tolist())
attacks    = sorted(df['attack'].dropna().unique().tolist())
metrics    = sorted(df['metric'].dropna().unique().tolist())
sparsities = sorted(df['sparsity'].dropna().unique().tolist())

dd_dataset = W.Dropdown(options=['(all)'] + datasets, value=(datasets[0] if datasets else None), description='dataset')
dd_attack  = W.Dropdown(options=attacks, value=(attacks[0] if attacks else None), description='attack')
dd_metric  = W.Dropdown(options=metrics, value=(metrics[0] if metrics else None), description='metric')
dd_spars   = W.Dropdown(options=sparsities, value=(sparsities[0] if sparsities else None), description='sparsity')
dd_sort    = W.Dropdown(options=['mean','median','n','std','ci95_lo','ci95_hi','iqr'], value='mean', description='sort by')
dd_asc     = W.Checkbox(value=False, description='ascending')

out = W.Output()

def refresh(*_):
    with out:
        out.clear_output()
        ds = None if dd_dataset.value == '(all)' else dd_dataset.value
        pvt, tbl = pivot_block(ds, dd_spars.value, dd_attack.value, dd_metric.value, dd_sort.value, dd_asc.value)
        if pvt.empty:
            display(HTML('<b>No rows match the current filters.</b>'))
            return
        display(HTML('<h3>Pivot: rows = (method, mode)</h3>'))
        display(pvt)
        display(HTML('<h4>Grouped rows</h4>'))
        display(tbl)

for w in [dd_dataset, dd_attack, dd_metric, dd_spars, dd_sort, dd_asc]:
    w.observe(refresh, names='value')

display(W.HBox([dd_dataset, dd_attack, dd_metric]))
display(W.HBox([dd_spars, dd_sort, dd_asc]))
display(out)
refresh()


Output()

In [ ]:
# Optional: export current pivot to CSV (run after setting widgets)
from datetime import datetime
def export_current():
    ds = None if dd_dataset.value == '(all)' else dd_dataset.value
    pvt, tbl = pivot_block(ds, dd_spars.value, dd_attack.value, dd_metric.value, dd_sort.value, dd_asc.value)
    if pvt.empty:
        print('Nothing to export'); return
    out_dir = Path('results/pivots'); out_dir.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime('%Y%m%d_%H%M%S')
    base = f"pivot_{ds or 'all'}_s{dd_spars.value}_{dd_attack.value}_{dd_metric.value}_{dd_sort.value}{'_asc' if dd_asc.value else '_desc'}_{ts}"
    pvt.to_csv(out_dir / f"{base}.csv")
    print('saved:', out_dir / f"{base}.csv")

# export_current()  # call to save current pivot
